In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-")
PHASE2_DIR = ROOT / "data_labeling" / "data_labels" / "GOES_2010_2018_NOAA_2019_2026"
NOAA_CL_DIR = ROOT / "data_labeling" / "data_labels" / "noaa_2010_2026" / "CL_Labels"
OUT_DIR = PHASE2_DIR / "CL_Labels_noaa_dates"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = PHASE2_DIR / "labels_goes_2010_2018_noaa_2019_2026_complete_binary.csv"

def parse_timestamp(series: pd.Series) -> pd.Series:
    ts_str = series.str.extract(r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})")[0]
    return pd.to_datetime(ts_str, format="%Y.%m.%d_%H.%M.%S")

master_df = pd.read_csv(MASTER_CSV)
master_df["timestamp"] = parse_timestamp(master_df["label"])
master_df = master_df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded Phase 2 master CSV: {len(master_df):,} rows")
print(f"Range: {master_df['timestamp'].min()} -> {master_df['timestamp'].max()}")
print(f"FL={int((master_df['goes_class'] == 1).sum()):,} ({master_df['goes_class'].mean()*100:.2f}%)")

Loaded Phase 2 master CSV: 128,116 rows
Range: 2010-12-06 07:00:00 -> 2026-07-31 23:00:00
FL=26,999 (21.07%)


In [4]:
STAGE1_FILES = ["Stage1_train.csv", "Stage1_holdout.csv"]

splits = {}
summary_rows = []

for fname in STAGE1_FILES:
    ref_df = pd.read_csv(NOAA_CL_DIR / fname)
    ref_ts = parse_timestamp(ref_df["label"])
    t_min, t_max = ref_ts.min(), ref_ts.max()

    sub = master_df[(master_df["timestamp"] >= t_min) & (master_df["timestamp"] <= t_max)].copy()
    splits[fname] = sub

    fl = int((sub["goes_class"] == 1).sum())
    nf = int((sub["goes_class"] == 0).sum())
    ref_fl = int((ref_df["goes_class"] == 1).sum())

    summary_rows.append({
        "split": fname,
        "phase2_total": len(sub),
        "phase2_FL": fl,
        "phase2_NF": nf,
        "phase2_FL%": round(fl / len(sub) * 100, 2) if len(sub) else 0.0,
        "noaa_total": len(ref_df),
        "noaa_FL": ref_fl,
        "noaa_FL%": round(ref_fl / len(ref_df) * 100, 2),
        "start": t_min,
        "end": t_max,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,split,phase2_total,phase2_FL,phase2_NF,phase2_FL%,noaa_total,noaa_FL,noaa_FL%,start,end
0,Stage1_train.csv,46637,8645,37992,18.54,46638,8623,18.49,2010-12-06 07:00:00,2017-01-06 03:00:00
1,Stage1_holdout.csv,8230,330,7900,4.01,8230,330,4.01,2017-01-07 04:00:00,2017-12-31 23:00:00


In [5]:
s1_train = splits["Stage1_train.csv"]
s1_holdout = splits["Stage1_holdout.csv"]

overlap = set(s1_train["label"]) & set(s1_holdout["label"])
gap_hours = (s1_holdout["timestamp"].min() - s1_train["timestamp"].max()).total_seconds() / 3600.0

print(f"Overlap count: {len(overlap)}")
print(f"Gap Stage1_train -> Stage1_holdout: {gap_hours:.1f} hours")
assert len(overlap) == 0, "Overlap found!"
assert gap_hours >= 25.0, f"Expected >= 25h gap, got {gap_hours:.1f}h"
print("Stage 1 verification passed!")

Overlap count: 0
Gap Stage1_train -> Stage1_holdout: 25.0 hours
Stage 1 verification passed!


In [6]:
for fname, df_out in splits.items():
    out_path = OUT_DIR / fname
    df_out[["label", "goes_class"]].to_csv(out_path, index=False)
    print(f"Saved {out_path} -> {len(df_out):,} rows")

Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage1_train.csv -> 46,637 rows
Saved /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels_noaa_dates/Stage1_holdout.csv -> 8,230 rows
